# Pichia-CLM Arch1 — Colab Training Notebook

**Replication of:** Narayanan & Love, PNAS 2026  
**Repo:** https://github.com/baka-44/PichiaCLM  

### Before running:
1. `Runtime → Change runtime type → GPU (T4 or A100)`
2. Run cells top to bottom
3. Model weights save to Google Drive — they persist after session ends

### Expected training time:
- T4 GPU  : ~45–60 min
- A100 GPU : ~15–20 min
- CPU only  : ~4–6 hrs (not recommended)

## Cell 1 — Mount Google Drive (persistent checkpoint storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/PichiaCLM/checkpoints'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')

## Cell 2 — Clone your fork and install dependencies

In [ ]:
import os

# ── Switch Keras to JAX backend BEFORE any keras/tf import ──────────────────
os.environ['KERAS_BACKEND'] = 'jax'
print('Keras backend set to: jax')

# Clone if fresh VM, pull if repo already exists
if os.path.exists('/content/PichiaCLM/.git'):
    !git -C /content/PichiaCLM pull
else:
    !git clone https://github.com/baka-44/PichiaCLM.git /content/PichiaCLM

%cd /content/PichiaCLM

# Upgrade the CUDA plugin to match the pre-installed JAX version.
# Colab has JAX 0.9.x but jax-cuda12-plugin stays at 0.7.x → PJRT mismatch.
!pip install -q -U jax-cuda12-plugin
!pip install -q scikit-learn seaborn

import jax
print('JAX version:', jax.__version__)
print('JAX devices:', jax.devices())   # Should show: [CudaDevice(id=0)]

import tensorflow as tf
print('TensorFlow:', tf.__version__)

## Cell 3 — Add scripts/ to path and import modules

In [ ]:
import sys
sys.path.insert(0, '/content/PichiaCLM/scripts')

from data_prep import prepare_data, DATA_DIR, AA_MAXLEN, CDS_MAXLEN
from model import build_training_model, build_inference_models, DEFAULT_HP, MAX_LENGTH, BATCH_SIZE, EPOCHS
from train import make_inputs_targets, make_dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, os

print('All imports OK')

## Cell 4 — Prepare data
Loads all 5 CSVs, filters sequences with N, does 80/20 protein-level split,
prepends 25× augmented single-codon pairs, tokenises and pads.

In [ ]:
data = prepare_data(
    data_dir='/content/PichiaCLM/Model_PichiaCLM/Training/AllData',
    verbose=True,
)

# 80/20 split within training data for train vs validation
n_total = len(data['AA_tr'])
n_train = int(n_total * 0.80)

train_inputs, train_targets = make_dataset(data['AA_tr'][:n_train], data['Cds_tr'][:n_train], shuffle=True)
val_inputs,   val_targets   = make_dataset(data['AA_tr'][n_train:], data['Cds_tr'][n_train:], shuffle=False)
test_inputs,  test_targets  = make_inputs_targets(data['AA_ts'], data['Cds_ts'])

print(f"\nTraining sequences   : {n_train:,}")
print(f"Validation sequences : {n_total - n_train:,}")
print(f"Test sequences       : {len(data['AA_ts']):,}")
print(f"\ntrain_inputs[0].shape : {train_inputs[0].shape}  (enc_aa)")
print(f"train_inputs[1].shape : {train_inputs[1].shape}  (dec_codon)")
print(f"train_inputs[2].shape : {train_inputs[2].shape}  (dec_aa)")

## Cell 5 — Build the model

In [ ]:
import jax

# Verify JAX can see the GPU
devices = jax.devices()
print('JAX devices:', devices)
gpu_devices = [d for d in devices if 'cuda' in str(d).lower() or 'gpu' in str(d).lower()]
if not gpu_devices:
    raise RuntimeError('No GPU detected by JAX — check Runtime → Change runtime type → GPU')
print(f'GPU confirmed: {gpu_devices[0]}')

# Build model (no tf.device needed — JAX auto-places on GPU)
import keras
model = build_training_model(DEFAULT_HP)

model.summary()
print(f'\nTotal parameters: {model.count_params():,}')
print(f'\nKeras backend in use: {keras.backend.backend()}')

## Cell 6 — Set up callbacks

In [ ]:
import keras
import os

ckpt_path = os.path.join(CHECKPOINT_DIR, 'pichia_clm_arch1_ep{epoch:03d}.weights.h5')

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=ckpt_path,
        save_weights_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(
        os.path.join(CHECKPOINT_DIR, 'training_history.csv'),
        append=True,
    ),
]

print('Callbacks set up. Checkpoints will be saved to Google Drive after each epoch.')

## Cell 7 — Train
Early stopping monitors `val_loss` with patience=5.  
Model weights from the best epoch are restored automatically.

In [ ]:
import time

# --- GPU execution check: time a single training batch ---
print("Timing one batch to confirm GPU vs CPU execution...")
t0 = time.time()
_ = model.train_on_batch(
    [x[:BATCH_SIZE] for x in train_inputs],
    [y[:BATCH_SIZE] for y in train_targets]
)
elapsed = time.time() - t0

if elapsed < 30:
    print(f"✓ GPU confirmed: 1 batch = {elapsed:.1f}s  (expected ~2–10s on A100)")
else:
    print(f"✗ Likely CPU: 1 batch = {elapsed:.1f}s  — stop and report before running fit()")
print()

# Full training — JAX auto-places on GPU, no tf.device needed
history = model.fit(
    train_inputs,
    train_targets,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(val_inputs, val_targets),
    callbacks=callbacks,
    verbose=1,
)

## Cell 8 — Plot training curves

In [ ]:
hist_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(hist_df['loss'],     label='Train loss')
axes[0].plot(hist_df['val_loss'], label='Val loss')
axes[0].set_title('Total loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Codon accuracy
codon_acc_col = [c for c in hist_df.columns if 'output_codon' in c and 'accuracy' in c and 'val' not in c]
val_codon_acc_col = [c for c in hist_df.columns if 'output_codon' in c and 'accuracy' in c and 'val' in c]

if codon_acc_col:
    axes[1].plot(hist_df[codon_acc_col[0]],     label='Train codon acc')
    axes[1].plot(hist_df[val_codon_acc_col[0]], label='Val codon acc')
    axes[1].set_title('Codon prediction accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_curves.png'), dpi=150)
plt.show()
print('Curves saved to Drive.')

## Cell 9 — Evaluate on held-out test proteins (80/20 split)

In [ ]:
print('Evaluating on held-out test proteins...')
results = model.evaluate(test_inputs, test_targets, batch_size=BATCH_SIZE, verbose=1)

print('\nTest results:')
for name, val in zip(model.metrics_names, results):
    print(f'  {name}: {val:.4f}')

## Cell 10 — Save final model weights + hyperparameters to Drive

In [ ]:
final_weights_path = os.path.join(CHECKPOINT_DIR, 'pichia_clm_arch1_final.weights.h5')
model.save_weights(final_weights_path)
print(f'Final weights saved: {final_weights_path}')

hp_path = os.path.join(CHECKPOINT_DIR, 'hyperparameters.json')
with open(hp_path, 'w') as f:
    json.dump({
        **DEFAULT_HP,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'epochs_run': len(history.history['loss']),
    }, f, indent=2)
print(f'Hyperparameters saved: {hp_path}')

print('\nAll done! Model weights are in Google Drive and will persist after session ends.')

## Cell 11 — (Optional) Resume training from a checkpoint
Run this cell instead of Cell 7 if your Colab session disconnected mid-training.

In [ ]:
# List available checkpoints
ckpts = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.h5')])
print('Available checkpoints:')
for c in ckpts:
    print(' ', c)

# Load the latest checkpoint
latest = os.path.join(CHECKPOINT_DIR, ckpts[-1])
model.load_weights(latest)
print(f'\nLoaded weights from: {latest}')
print('Now re-run Cell 7 to continue training from this epoch.')